# Single Agent - Framework




## Install dependencies
Run this once in a fresh environment.


In [2]:
# %pip -q install langchain-openai python-dotenv
# %pip install -U langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Imports

In [ ]:
# from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage

c:\Users\chunh\Documents\2606_mobile-class\2-YK_sample_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Load environment variables - please read instructions carefully

In [ ]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
# load_dotenv()

True

In [5]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

## 3) System prompt

In [6]:
SYSTEM = """You are a travel cost estimation agent.

Rules:
- Ask clarifying questions only when essential info is missing.
- Do not invent facts. Use search tool for fresh facts when needed.
- When user asks "total cost", ALWAYS call estimate_trip_cost with the latest trip parameters from conversation.
- If user says "add additional one day trip", interpret as +1 day unless they explicitly say it replaces an existing day.
- If a tool fails, read the tool error carefully, correct the arguments, and retry once. NEVER ask the user — fix it yourself.
- If the retry still fails, explain the issue clearly to the user.

Output format:
1) Total cost (with assumptions)
"""


## 4) Tool - estimate trip cost

In [7]:
@tool
def estimate_trip_cost(days: int, travelers: int, comfort: str = "mid"):
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """

    comfort = comfort.lower().strip()

    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be budget, mid, or premium")

    rates = {
        "budget": 120,
        "mid": 270,
        "premium": 590,
    }

    return {
        "days": days,
        "travelers": travelers,
        "comfort": comfort,
        "total_estimate": days * travelers * rates[comfort],
    }

In [12]:
class PrintMessagesAfterModel(AgentMiddleware):
    def after_model(self, state, runtime):
        print("\n================= Messages before model call =================")
        for msg in state["messages"]:
            print(f"\n{msg.type}")
            print(msg)
        print("================================================================\n")

In [13]:
class PrintMessagesBeforeModel(AgentMiddleware):
    def before_model(self, state, runtime):
        print("\n================= Messages after model =================")
        for msg in state["messages"]:
            print(f"\n{msg.type}")
            print(msg)
        print("================================================================\n")


## 5) Agent


In [17]:
from langchain_openai import ChatOpenAI

local_llm = ChatOpenAI(
    model="qwen3",                 
    temperature=0,
    openai_api_key="ollama",            
    openai_api_base="http://localhost:11434/v1"
)

agent = create_agent(
    model=local_llm,                    
    tools=[estimate_trip_cost],
    middleware=[PrintMessagesBeforeModel(), PrintMessagesAfterModel()],
    system_prompt=SYSTEM
)

# agent = create_agent(
#     model="gpt-4.1-mini",
#     tools=[estimate_trip_cost],
#     # middleware=[PrintMessagesBeforeModel(), PrintMessagesAfterModel()],
#     system_prompt=SYSTEM
# )

## 6) Run

In [ ]:
msg = "How much is a 2-day Tokyo trip for 2 adults with mid comfort?"
# agent.invoke({"messages": [{"role": "user", "content": msg}]})
result = agent.invoke({ "messages": [HumanMessage(content=msg)]})


================= Messages after model =================

human
content='How much is a 2-day Tokyo trip for 2 adults with mid comfort?' additional_kwargs={} response_metadata={} id='19b043d5-9860-451d-b65a-3e8d0c1cc0b8'


================= Messages before model call =================

human
content='How much is a 2-day Tokyo trip for 2 adults with mid comfort?' additional_kwargs={} response_metadata={} id='19b043d5-9860-451d-b65a-3e8d0c1cc0b8'

ai
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 327, 'total_tokens': 500, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-8', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f20d8-80f4-7d50-9d7d-78001f286ac8-0' tool_calls=[{'name': 'estimate_trip_cost', 'args': {'comfort': 'mid', 'days': 2, 'travelers': 2}, 'id': 'call_b62xyh99', '

In [16]:
print(result)

{'messages': [HumanMessage(content='How much is a 2-day Tokyo trip for 2 adults with mid comfort?', additional_kwargs={}, response_metadata={}, id='871e8ff8-b72a-4e9e-97cc-ac2691e0abe2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 327, 'total_tokens': 500, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-252', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f20d6-f749-7613-a436-d4fccc62fe36-0', tool_calls=[{'name': 'estimate_trip_cost', 'args': {'comfort': 'mid', 'days': 2, 'travelers': 2}, 'id': 'call_ajg0kyat', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 327, 'output_tokens': 173, 'total_tokens': 500, 'input_token_details': {}, 'output_token_details': {}}), ToolMessage(content='{"days": 2, "travelers": 2, "comfort": "mid", 